In [0]:
dbutils.widgets.text("catalog_parameter", "dev_catalog")
target_catalog = dbutils.widgets.get("catalog_parameter")
print(target_catalog)

In [0]:
from pyspark.sql import functions as F

# 1. Grab the environment catalog parameter dynamically from the Asset Bundle
dbutils.widgets.text("catalog_parameter", "dev_catalog")
target_catalog = dbutils.widgets.get("catalog_parameter")

print(f"🚀 Initializing raw sales extraction targeting catalog: {target_catalog}")

# 2. Generate clean mock retail sales data for our sample project
mock_data = [
    (1001, "2026-07-17", "Electronics", 450.00, "NY"),
    (1002, "2026-07-17", "Apparel", 29.99, "CA"),
    (1003, "2026-07-17", "Home Goods", 125.50, "TX"),
    (1004, "2026-07-17", "Electronics", 899.00, "FL")
]
columns = ["transaction_id", "date", "category", "amount", "state"]

df_raw = spark.createDataFrame(mock_data, schema=columns)

# 3. Add operational metadata (ingestion timestamp) so we know exactly when data landed
df_bronze = df_raw.withColumn("ingested_at", F.current_timestamp())

# 4. Write out the data into our Unity Catalog Bronze layer
# It will land in dev_catalog.bronze.raw_sales, uat_catalog.bronze.raw_sales, etc.
target_table = f"{target_catalog}.bronze.raw_sales"

print(f"📥 Writing raw extractions to Delta table: {target_table}")
df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(target_table)

print("✅ Bronze data extraction completed successfully!")